<a href="https://colab.research.google.com/github/AdityaJana011/DS635-MLS/blob/main/docs/labs/Lab13_14_decoding_strategies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 13/14 — Decoding strategies, measured

Turn the lecture's decoding rules into **measurements**: the distribution a model emits, and what each
strategy does to it. Read the [lab brief](Lab13_14.md) first. You are marked on **relationships between
your own numbers** — the sampling parts are seeded from your roll number, so your numbers are yours.

**Before you run anything:** set your identity in the next cell. Fill every prediction and explanation cell. Run top to bottom,
then run the final export cell and submit the two files it names.

In [8]:
ROLL_NUMBER = "202518035"      # <- your roll number, e.g. "202512345"
NAME        = "Aditya Jana"      # <- your name

# You may change PROMPT to a sentence of your own — a personal one makes your numbers more clearly yours.
PROMPT = "The butterfly counts not months but moments, and has time enough."

assert ROLL_NUMBER and NAME, "Set ROLL_NUMBER and NAME before running the rest."

## Setup

In [9]:
import json, platform, sys, hashlib
from pathlib import Path
import torch, torch.nn.functional as F
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()

MODEL = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL).eval()
model.generation_config.pad_token_id = tokenizer.eos_token_id

SEED = int(hashlib.sha256(ROLL_NUMBER.encode()).hexdigest(), 16) % (2**31)   # your sampling seed
inputs = tokenizer(PROMPT, return_tensors="pt")
PLEN = inputs["input_ids"].shape[1]

def next_logits(prompt):
    ids = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        return model(**ids).logits[0, -1]

RESULTS = {}
print("model", MODEL, "| vocab", model.config.vocab_size, "| seed", SEED)

model gpt2 | vocab 50257 | seed 214463790


---
## Part 1 — The distribution and greedy

📝 **Predict.** The model scores every token in a ~50k vocabulary. Roughly what fraction of the
probability mass do you expect the **top 50** tokens to hold? Run greedy decoding twice — will the two
outputs be identical? Will greedy's first token be the **argmax** of the distribution?


1. Generally in the distribution of natural language, token frequency follows a power-law decay or Zipfian behaviour. So a handful of plausible continuations take up the vast majority of the confidence while the remaining 50K tokens each get tiny fractions of probability. So my guess is that the top 50 tokens will at least get 50% of the probability mass, or maybe even more.

2. The outputs for the two runs g1 and g2 should be identical, as Hugging Face's built-in model.generate() with do_sample=False makes it execute an internal torch.argmax() at every step. Since we pass the exact same input string into this loop twice, and at every iteration it strictly executes argmax, the two outputs will be identical.

3. Yes, greedy's first token will be the argmax. The selection rule is just argmax at every step, so right at step 1 it simply takes the argmax of the logits from the initial prompt.

In [10]:
logits = next_logits(PROMPT)
probs = F.softmax(logits, dim=-1)
top50_mass = torch.topk(probs, 50).values.sum().item()

g1 = model.generate(**inputs, max_new_tokens=40, do_sample=False)[0, PLEN:].tolist()
g2 = model.generate(**inputs, max_new_tokens=40, do_sample=False)[0, PLEN:].tolist()

RESULTS["dist"] = dict(vocab_size=int(model.config.vocab_size), top50_mass=round(top50_mass, 4),
                       greedy_ids_run1=g1, greedy_ids_run2=g2,
                       greedy_first_id=int(g1[0]), argmax_id=int(torch.argmax(logits)))
print(f"top-50 mass            : {top50_mass:.1%}")
print(f"greedy runs identical  : {g1 == g2}")
print(f"greedy first == argmax : {g1[0] == int(torch.argmax(logits))}")
print("greedy text:", tokenizer.decode(g1))

top-50 mass            : 75.8%
greedy runs identical  : True
greedy first == argmax : True
greedy text: 

"I'm not sure if I'm going to be able to do it," he said. "I'm not sure if I'm going to be able to do it. I'm not


📝 **Explain.** Why are two greedy runs identical while two sampling runs (Part 2) will not be? Why
must greedy's first token equal the argmax? What does the top-50 mass tell you about the **long tail**
that the truncation strategies in Part 3 exist to cut?

<br><br>

1. The two greedy runs produce identical outputs because the forward pass maps the fixed prompt to identical logits, and greedy decoding applies torch.argmax() which is a purely deterministic function with no random state.  It will always return the exact same token index for the same scores. In contrast, sampling treats the softmax outputs as a categorical distribution and draws tokens using a pseudo-random number generator. Even though the probability distribution is identical for the same prompt, the random draw can pick different candidates. Because autoregressive generation appends each chosen token back into the input context, the very first step where the two sampling runs diverge alters all future forward passes, permanently splitting their generation trajectories.


2. Greedy's first token must equal the argmax because greedy decoding operates purely as a local, step-by-step decision rule. At step 1, the model evaluates only the initial prompt to produce its first logit vector. The greedy rule takes the index corresponding to the largest value in that vector. Because no external state, temperature scaling, or sampling alters this operation, the first generated token is, by definition, the exact argmax of the model's initial distribution.

3. The top-50 mass shows that natural language probability is heavily concentrated in a tiny fraction of the vocabulary, leaving the remaining ~50K tokens to share only a small slice of the distribution. However, because softmax assigns a non-zero probability to every entry in the vocabulary, the cumulative mass of the rest of the individually negligible tail tokens adds up to a meaningful total risk. In untruncated sampling, rolling against this full distribution across a long sequence makes it likely that an irrelevant or nonsensical tail token will eventually be selected. Once that low-quality token enters the context, the model conditions all future generation on it, derailing the text. Truncation strategies like top-k and top-p exist to explicitly zero out this collective tail before the draw occurs.

---
## Part 2 — Temperature

📝 **Predict.** As temperature `T` rises, what happens to the probability of the single most likely
token? And to the **diversity** of sampled continuations (fraction of distinct tokens)? Predict the
*direction* of each change before you measure.

<br>
<br>

As temperature $T$ rises, the probability of the single most likely token will decrease. Increasing $T$ scales down the magnitude of all logits, which diminishes the relative difference $(z_{\max} - z_j)/T$ between the top candidate and competitors. This flattens the softmax distribution toward uniform, dispersing probability mass away from the peak across the rest of the vocabulary.

Conversely, the diversity of sampled continuations (fraction of distinct tokens) will increase. Because the distribution becomes flatter, non-top tokens gain substantial probability mass, making it far more likely for the random draw to select different tokens across repeated runs rather than repeatedly collapsing onto the top few options.

In [11]:
Ts = [0.5, 1.0, 2.0]
p_top = [F.softmax(logits / T, dim=-1).max().item() for T in Ts]     # same logits, reshaped

def distinct_ratio(T, n=16, k=20):
    toks = []
    for i in range(n):
        torch.manual_seed(SEED + i)                                 # seeded from your roll number
        out = model.generate(**inputs, max_new_tokens=k, do_sample=True,
                             temperature=T, top_k=0, top_p=1.0)
        toks += out[0, PLEN:].tolist()
    return len(set(toks)) / len(toks)

div_Ts = [0.7, 1.0, 1.5]
distinct = [round(distinct_ratio(T), 4) for T in div_Ts]

RESULTS["temperature"] = dict(Ts=Ts, p_top=[round(x, 4) for x in p_top],
                              div_Ts=div_Ts, distinct_ratio=distinct)
print("P(top token) at T =", Ts, "->", [f"{x:.2%}" for x in p_top])
print("distinct-token ratio at T =", div_Ts, "->", distinct)

P(top token) at T = [0.5, 1.0, 2.0] -> ['83.46%', '26.23%', '1.16%']
distinct-token ratio at T = [0.7, 1.0, 1.5] -> [0.5847, 0.6844, 0.9049]


📝 **Explain.** Temperature divides the logits before softmax. Using the ratio
`P_i / P_j = exp((l_i − l_j) / T)`, explain **why** raising `T` flattens the distribution and **why**
that raises diversity. What happens in the limit `T → 0`?

*(your explanation here)*

---
## Part 3 — Truncation: top-k vs nucleus (top-p)

📝 **Predict.** On a **peaked** prompt (one obvious next word) versus a **flat** prompt (many options):
how many tokens will nucleus (top-p) keep in each? How many will top-k keep in each? Which one *adapts*?

<br>

From the ratio:

$$
\frac{P_i}{P_j} = \exp\left(\frac{l_i - l_j}{T}\right)
$$

As $T$ rises, $(l_i - l_j) / T \to 0$, so $P_i / P_j \to 1$ for all token pairs. This reduces the gap between top and bottom candidates, spreading probability mass evenly across the vocabulary and flattening the distribution toward uniform.

---

That flattening boosts diversity because lower-ranked tokens get real probability mass instead of near-zero values. The random draw can now pick something else instead of picking from the top few tokens anymore, so it explores a wider vocabulary across runs, raising the unique-token ratio.

---

As $T \to 0$, for the top logit $l_{\max}$ and any competitor $l_j$, $(l_{\max} - l_j) / T \to +\infty$. The ratio explodes to infinity, pushing $P_{\max} \to 1$ and all other probabilities to $0$. The distribution collapses to a single point like a dirac delta function at the index of $l_{\max}$, turning sampling into a deterministic argmax (greedy decoding).

In [12]:
PEAKED = "The United States of"     # one obvious next token
FLAT   = "My favourite food is"     # many reasonable ones
P, K = 0.9, 50

def topp_stats(lg, p):
    s, _ = torch.sort(F.softmax(lg, dim=-1), descending=True)
    n = int((torch.cumsum(s, dim=-1) < p).sum()) + 1               # smallest set reaching p
    kept = s[:n].sum().item()
    kept_minus_last = s[:n-1].sum().item() if n > 1 else 0.0
    return n, kept, kept_minus_last

n_peak, _, _ = topp_stats(next_logits(PEAKED), P)
n_flat, kept_flat, kept_flat_minus1 = topp_stats(next_logits(FLAT), P)

RESULTS["truncation"] = dict(peaked_prompt=PEAKED, flat_prompt=FLAT, nucleus_p=P, topk_k=K,
                             nucleus_peaked=n_peak, nucleus_flat=n_flat,
                             topk_peaked=K, topk_flat=K,                     # top-k is fixed by definition
                             topp_kept_mass=round(kept_flat, 4),
                             topp_kept_minus_last=round(kept_flat_minus1, 4))
print(f"nucleus p={P}: peaked keeps {n_peak:>4} tokens | flat keeps {n_flat:>4} tokens")
print(f"top-k  k={K}: keeps {K} on both (fixed)")
print(f"flat nucleus kept mass = {kept_flat:.3f} (>= {P}?) ; without its last token = {kept_flat_minus1:.3f} (< {P}?)")

nucleus p=0.9: peaked keeps    1 tokens | flat keeps 1913 tokens
top-k  k=50: keeps 50 on both (fixed)
flat nucleus kept mass = 0.900 (>= 0.9?) ; without its last token = 0.900 (< 0.9?)


📝 **Explain.** Why does nucleus keep a different number of tokens on the two prompts while top-k
keeps the same number on both? Name the failure each one fixes — and the failure each one still has.

<br>

Top-k keeps a fixed number of tokens because its cutoff rule is strictly based on rank count, completely ignoring how the probabilities are actually distributed. Whether the model is 99% confident or totally uncertain, it just slices off the top $k$ items.

Nucleus (top-p), on the other hand, cuts off based on cumulative probability mass ($\sum P_i \ge p$). When the distribution is peaked, a single token already holds over $90\%$ of the mass, so the loop stops right there at 1 token. But when the distribution is flat and entropy is high, each token only holds a tiny fraction of probability, so it has to sum across nearly 2,000 tokens before hitting that $0.9$ threshold.

---

* **Top-k:**
  * **Fixes:** The tail-risk of pure sampling by setting a hard limit, preventing the model from ever drawing from the tens of thousands of irrelevant tokens in the tail.
  * **Remaining failure:** It has no awareness of model confidence. On peaked steps, it forces k-1 low-probability junk into the pool and renormalizes it; on flat steps, it amputates hundreds of perfectly valid continuations just because they fall past rank $k$.

* **Nucleus (Top-p):**
  * **Fixes:** Top-k's rigid cutoff problem by letting the candidate pool dynamically grow or shrink based on how confident the model actually is.
  * **Remaining failure:** If the distribution gets extremely flat (like with high temperature or confusing context), accumulating up to $p = 0.9$ can pull in thousands of tail tokens, letting noisy, low-quality words creep back into the sample.

---
## Part 4 — Beam vs greedy (the honest one)

Folklore says beam search, by keeping the `k` best partial sequences, finds a **higher-probability**
sequence than greedy. Measure whether it actually does on *your* run.

📝 **Predict.** Will beam's total sequence log-probability beat greedy's? Why might a *heuristic*
(non-exhaustive) search fail to?

<br>

Beam's total sequence log-probability should beat (or at least match) greedy. Greedy is just beam search with width=1. With multiple beams, it can pick a slightly lower-probability token right now if that path leads to much higher probability tokens later on.

---

Still, beam is just a heuristic and not a full brute-force search over the whole tree, so it can fail:

* **Early pruning:** Beam only keeps the top B paths at each step and drops the rest. If the actual best sequence starts with an unusual word that falls outside the top B early on, beam drops it immediately and can never backtrack.

* **Length difference:** Log-probs are always negative numbers, so every extra token we add makes the total sum smaller (more negative). If greedy stops earlier or if beam optimizes for average score per token instead of raw total sum, greedy might end up with a higher raw total log-prob just because it's shorter.

In [13]:
def seq_logprob(full_ids):
    with torch.no_grad():
        lp = F.log_softmax(model(full_ids).logits[0, :-1], dim=-1)
    tgt = full_ids[0, 1:]
    return lp[PLEN-1:].gather(1, tgt[PLEN-1:].unsqueeze(1)).sum().item()   # log-prob of the generated tokens

greedy_out = model.generate(**inputs, max_new_tokens=30, do_sample=False)
beam_out   = model.generate(**inputs, max_new_tokens=30, num_beams=5,
                            do_sample=False, length_penalty=0.0, early_stopping=False)
gl, bl = seq_logprob(greedy_out), seq_logprob(beam_out)

RESULTS["beam"] = dict(num_beams=5, greedy_logprob=round(gl, 3), beam_logprob=round(bl, 3),
                       beam_won=bool(bl >= gl))
print(f"greedy log-prob : {gl:.2f}")
print(f"beam   log-prob : {bl:.2f}")
print("beam won (>= greedy)?", bl >= gl)

greedy log-prob : -33.72
beam   log-prob : -29.76
beam won (>= greedy)? True


📝 **The paragraph that carries this part.** Did beam beat greedy on *your* run? Beam is not
exhaustive — it prunes low-scoring prefixes. Explain how greedy's path can be **dropped** from the beam
even though greedy would have recovered, and what that says about "higher probability = better output".
If beam *did* win, say what it found that greedy could not see.

<br>

Yes, beam beat greedy on my run (-29.76 vs -33.72). Greedy is myopic and blindly locks into the argmax at step 1 without looking ahead, which trapped it on a path where later tokens had poor probabilities. Beam search kept 5 tracks alive, so it could pick a slightly worse token early on to unlock a path where downstream tokens had much higher probabilities, giving a better total sum over the 30 tokens.

---

Even though greedy takes the best token locally, its whole path can easily get kicked out of the beam. If other paths branch into multiple strong candidates at step $t$, they can take up all top 5 spots in the beam pool. Greedy's path drops to rank 6 or lower and gets pruned immediately. Since beam search cannot backtrack, even if greedy's branch would have recovered later with a string of near 100% confidence tokens, it's gone for good.

---

This shows that "higher probability = better output" isn't always true for text generation. Autoregressive models often give the highest probabilities to repetitive loops, generic phrases, or dull text because repetition artificially inflates confidence. So a sequence with a higher log-prob can actually sound worse or more robotic to a human than one with a slightly lower score.

📝 **Closing — choose and justify.** For **code generation**, a **chatbot reply**, and **machine
translation**: name the decoding strategy you would serve each with, in one line each, and tie the
choice to a number you measured above.

<br>

* **Code generation:Greedy (or low-T sampling with small top-k)**, because code needs strict syntactic correctness and zero hallucination; at $T=0.5$ the top token took **83.46%** of the probability mass, which keeps generation focused on correct syntax instead of taking random risks.

* **Chatbot reply: Nucleus sampling ($top\text{-}p \approx 0.9$ with $T \approx 0.7$–$1.0$)**, because conversations need natural phrasing without going crazy; our distinct ratio was **0.58–0.68** here, and nucleus dynamically filtered from **1 token** on confident steps to **1,913 tokens** on flat steps so it stays coherent without repeating itself.

* **Machine translation: Beam search (width 4–5)**, because translation has a fixed target meaning where future words depend heavily on sentence-level planning; beam achieved a much better total log-prob (**-29.76 vs -33.72**) by looking past myopic first-token choices to find a globally better sequence.

---
## Submit

Run this last. It writes `submission_lab13_14_<roll>.json`. Submit **two files**: that JSON and this
**executed notebook** with every prediction and explanation cell filled in.

In [15]:
env = dict(platform=platform.platform(), python=sys.version.split()[0],
           torch=torch.__version__, transformers=transformers.__version__,
           model=MODEL, seed=SEED, prompt=PROMPT, linux=sys.platform.startswith("linux"))
sub = dict(roll=ROLL_NUMBER, name=NAME, env=env, results=RESULTS)

out = Path(f"submission_lab13_14_{ROLL_NUMBER}.json")
out.write_text(json.dumps(sub, indent=2))
print("wrote", out)
print("  parts recorded:", list(RESULTS))
assert set(RESULTS) >= {"dist", "temperature", "truncation", "beam"}, "run every part before exporting"

wrote submission_lab13_14_202518035.json
  parts recorded: ['dist', 'temperature', 'truncation', 'beam']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')